## M3 — Índice F-of-M
Construimos el índice de atractivo territorial para rankear las 36.334 secciones censales.
F-of-M(s) = 0.30·Solvencia + 0.25·Demanda + 0.25·Soledad + 0.15·Concentración + 0.05·Futuro

In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df     = pd.read_csv('dataset_maestro.csv',
                     dtype={'seccion_censal': str},
                     encoding='utf-8-sig')
df_muni = pd.read_csv('municipios_limpios.csv')

print(f"Secciones cargadas: {df.shape[0]:,}")
print(f"Municipios cargados: {df_muni.shape[0]:,}")

Secciones cargadas: 36,334
Municipios cargados: 8,124


### Paso 1 — Normalización min-max de las variables

In [2]:
def normalizar(serie):
    minimo = serie.min()
    maximo = serie.max()
    return (serie - minimo) / (maximo - minimo)

# Normalizar cada variable
df['n_renta']          = normalizar(df['renta_hogar_2024'])
df['n_vol_75plus']     = normalizar(df['vol_75plus'])
df['n_pct_75plus']     = normalizar(df['pct_75plus'])
df['n_ratio_viudez']   = normalizar(df['ratio_viudez'])
df['n_cohorte_futura'] = normalizar(df['cohorte_futura'])

print("Variables normalizadas (rango 0-1):")
cols_n = ['n_renta', 'n_vol_75plus', 'n_pct_75plus',
          'n_ratio_viudez', 'n_cohorte_futura']
print(df[cols_n].describe().round(3))

Variables normalizadas (rango 0-1):
         n_renta  n_vol_75plus  n_pct_75plus  n_ratio_viudez  n_cohorte_futura
count  35012.000     36334.000     36334.000       36334.000         36334.000
mean       0.258         0.051         0.191           0.071             0.042
std        0.097         0.028         0.105           0.028             0.024
min        0.000         0.000         0.000           0.000             0.000
25%        0.191         0.032         0.121           0.055             0.028
50%        0.254         0.049         0.176           0.067             0.040
75%        0.298         0.067         0.239           0.082             0.054
max        1.000         1.000         1.000           1.000             1.000


### Paso 2 — Construir el índice F-of-M

In [3]:
# Pesos originales del equipo
# Solvencia:     0.30
# Demanda:       0.25 (vol + pct al 50% cada uno)
# Soledad:       0.25
# Concentración: 0.15 → pendiente de datos externos
# Futuro:        0.05

# Versión actual sin Concentración — pesos redistribuidos proporcionalmente
W_SOLVENCIA = 0.35   # 0.30 + parte de 0.15
W_VOL       = 0.15   # mitad del componente demanda
W_PCT       = 0.15   # mitad del componente demanda
W_SOLEDAD   = 0.30   # 0.25 + parte de 0.15
W_FUTURO    = 0.05

print("Pesos aplicados:")
print(f"  Solvencia:  {W_SOLVENCIA}")
print(f"  Volumen:    {W_VOL}")
print(f"  Porcentaje: {W_PCT}")
print(f"  Soledad:    {W_SOLEDAD}")
print(f"  Futuro:     {W_FUTURO}")
print(f"  TOTAL:      {W_SOLVENCIA+W_VOL+W_PCT+W_SOLEDAD+W_FUTURO}")

Pesos aplicados:
  Solvencia:  0.35
  Volumen:    0.15
  Porcentaje: 0.15
  Soledad:    0.3
  Futuro:     0.05
  TOTAL:      1.0


### Paso 3 — Calcular la puntuación final

In [4]:
df['fofm'] = (
    W_SOLVENCIA * df['n_renta'].fillna(0) +
    W_VOL       * df['n_vol_75plus'] +
    W_PCT       * df['n_pct_75plus'] +
    W_SOLEDAD   * df['n_ratio_viudez'] +
    W_FUTURO    * df['n_cohorte_futura']
)

# Aplicar filtro duro — secciones no solventes = 0
df.loc[df['solvente'] == 0, 'fofm'] = 0

print("=== DISTRIBUCIÓN DEL ÍNDICE F-of-M ===")
print(f"  Secciones con fofm > 0:   {(df['fofm'] > 0).sum():,}")
print(f"  Secciones con fofm = 0:   {(df['fofm'] == 0).sum():,}")
print(f"\n  Media:   {df[df['fofm']>0]['fofm'].mean():.4f}")
print(f"  Máximo:  {df['fofm'].max():.4f}")
print(f"  Mínimo (>0): {df[df['fofm']>0]['fofm'].min():.4f}")

df[['seccion_censal', 'renta_hogar_2024',
    'vol_75plus', 'ratio_viudez', 'fofm']].sort_values(
    'fofm', ascending=False).head(10)

=== DISTRIBUCIÓN DEL ÍNDICE F-of-M ===
  Secciones con fofm > 0:   24,438
  Secciones con fofm = 0:   11,896

  Media:   0.1639
  Máximo:  0.4469
  Mínimo (>0): 0.0864


,seccion_censal,renta_hogar_2024,vol_75plus,ratio_viudez,fofm
20622,2811501002,83812.0,229.0,4.416667,0.446894
20621,2811501001,83812.0,250.0,3.794118,0.436807
20651,2811502001,83812.0,183.0,3.000000,0.426819
20652,2811502002,83812.0,222.0,3.027027,0.425735
23936,3123202006,33555.0,50.0,25.000000,0.424879
20659,2811502009,83812.0,217.0,2.200000,0.423735
20624,2811501005,83812.0,141.0,2.062500,0.419078
20626,2811501007,83812.0,123.0,2.120000,0.418042
20628,2811501009,83812.0,142.0,1.718750,0.414715
20627,2811501008,83812.0,179.0,1.313725,0.414234


### Paso 4 — Añadir nombres de municipio y crear ranking

In [ ]:
# Añadir nombre de municipio
df = df.merge(
    df_muni[['CPRO', 'CMUN', 'NOMBRE']],
    on=['CPRO', 'CMUN'],
    how='left'
)

# Crear ranking
df['ranking'] = df['fofm'].rank(ascending=False, method='min').astype(int)

# Vista del top 20
top20 = df[df['fofm'] > 0].sort_values('fofm', ascending=False).head(20)

print("=== TOP 20 SECCIONES ===")
print(top20[['ranking', 'seccion_censal', 'NOMBRE',
             'renta_hogar_2024', 'vol_75plus',
             'ratio_viudez', 'fofm']].to_string(index=False))

['CODAUTO', 'CPRO', 'CMUN', 'DC', 'NOMBRE']
   CODAUTO  CPRO  CMUN  DC               NOMBRE
0       16     1    51   3  Agurain/Salvatierra
1       16     1     1   4     Alegría-Dulantzi
2       16     1     2   9              Amurrio
